# 封裝 -- 保護你的資料

## 學習目標

- 理解為什麼需要封裝
- 分辨 Python 的三種存取層級：public、_protected、__private
- 理解 name mangling 機制
- 學會用 `@property` 控制屬性的讀寫

---

## 為什麼需要封裝？

沒有封裝的程式碼，任何人都能直接修改資料：

In [ ]:
class BankAccount:
    def __init__(self, balance):
        self.balance = balance

acc = BankAccount(1000)
acc.balance = -999999   # 餘額變成負的？沒人阻止你

print(acc.balance) # 再呼叫一次就變成 -999999

-999999

封裝的目的：**防止物件進入不合理的狀態**。

---

## Python 的三種存取層級

Python 沒有像 Java 那樣的強制存取控制，而是靠**命名慣例**。

In [1]:
class MyClass:
    def __init__(self):
        self.public = "誰都能存取"          # 公開
        self._protected = "請不要直接碰我"    # 保護（慣例）
        self.__private = "外面很難碰到我"     # 私有（name mangling）

### 實際測試

In [ ]:
obj = MyClass()
print(obj.public)        # OK
print(obj._protected)    # OK（Python 不會擋，但不建議）
# print(obj.__private)   # AttributeError!
# 兩個底線__ 寫強制死
print(obj._MyClass__private)  # OK（這就是 name mangling） 加一個 _MyClass__private 還是叫的到
# 要先知道 Class 的名字



誰都能存取
請不要直接碰我
外面很難碰到我


Hint: name mangling 是 Python 對「類別內、雙底線開頭」名稱做的自動改名機制，目的是避免子類別不小心覆蓋它。

---

## Name Mangling 機制

雙底線開頭的屬性，Python 會自動改名：

```
__xxx  →  _ClassName__xxx
```

這不是真正的「私有」，而是**防止意外碰到**。你還是能透過改名後的名字存取，但這麼做表示你知道自己在幹嘛。

In [ ]:
class Secret:
    def __init__(self):
        self.__code = 1234

s = Secret()
# print(s.__code)             # AttributeError
print(s._Secret__code)        # 1234（刻意繞過）

---

## 用 @property 控制存取

`@property` 讓你把方法偽裝成屬性，在讀取和寫入時加上邏輯。

### 基本用法

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance 
    @property
    def balance(self):
        """讀取餘額"""
        return self._balance
    
# 有 @property 多加一層 Buff 多一層保護
# 但使用者無法感覺到


In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance     # 用 _ 表示不要直接碰
        # 對內是 顯示 _balance , 但對外是顯示 balance
    @property
    def balance(self):
        """讀取餘額"""
        return self._balance

    @balance.setter
    # 公版寫法, 直接記
    def balance(self, value):#　這個　balance 是指 property 裡的 balance 
        """設定餘額（含驗證）"""
        if value < 0:
            raise ValueError("餘額不能為負數")
        self._balance = value

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("存款金額必須為正")
        self._balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("提款金額必須為正")
        if amount > self._balance:
            raise ValueError("餘額不足")
        self._balance -= amount
    
    # 從 @property 開始的 ._balance 都是指向 __init__ 的._balance

    # @property 和 @balance.setter 這兩個是模板 
    # 知道是 getter 和 setter 就知道
    # 了解之後 , 丟給 AI 後, 特別說明是 getter 和 setter

### 使用起來像普通屬性

In [11]:
acc = BankAccount("小明", 1000)
print(acc.balance)     # 1000  ← 觸發 @property getter
acc.balance = 500      # OK    ← 觸發 @balance.setter
# acc.balance = -100   # ValueError: 餘額不能為負數

#這是呼叫 值
acc.balance   #  這邊是呼叫被 @property 變形的 .balance

acc.deposit(200)
print(acc.balance)     # 700
# acc.withdraw(800)    # ValueError: 餘額不足

1000
700


---

## 唯讀屬性

只定義 getter、不定義 setter，就是唯讀：

In [ ]:
class Circle:
    def __init__(self, radius):
        self._radius = radius

    @property
    def area(self):
        return 3.14159 * self._radius ** 2

c = Circle(5)
print(c.area)    # 78.53975
# c.area = 100   # AttributeError: can't set attribute
#　把上面一行的 # 去掉,就會發現是不給直接改的

IndentationError: unexpected indent (1692422399.py, line 11)

---

## 完整範例：溫度轉換

In [ ]:
class Temperature:
    def __init__(self, celsius=0):
        self.__celsius = celsius    # 觸發 setter 驗證

    @property
    def celsius(self):
        return self.__celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("溫度不能低於絕對零度")
        #　先擋掉錯誤的值
        self.__celsius = value

    @property
    def fahrenheit(self):
        return self.__celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = (value - 32) * 5 / 9   # 會觸發 celsius 的驗證

t = Temperature(100)
print(t.fahrenheit)    # 212.0
t.fahrenheit = 32 #　這是在　call @fahrenheit.setter
print(t.celsius)       # 0.0

212.0
0.0
50


---

## 存取層級比較表

| 命名 | 層級 | 外部可存取？ | 用途 |
|------|------|-------------|------|
| `name` | 公開 | 可以 | 一般屬性，對外開放 |
| `_name` | 保護（慣例） | 可以，但不建議 | 內部使用，提醒別人不要直接碰 |
| `__name` | 私有（mangling） | 需用 `_Class__name` | 防止子類別意外覆蓋 |
| `@property` | 受控存取 | 透過 getter/setter | 需要驗證或計算的屬性 |

---

## 本節重點

| 概念 | 說明 |
|------|------|
| 封裝的目的 | 防止物件進入不合理的狀態 |
| `_xxx` | 慣例上的保護屬性，Python 不強制 |
| `__xxx` | 觸發 name mangling，變成 `_Class__xxx` |
| `@property` | 把方法偽裝成屬性，加入讀寫控制 |
| 唯讀屬性 | 只定義 getter，不定義 setter |
| 驗證邏輯 | 放在 setter 中，確保資料永遠合法 |

---

> **下一篇：** [04_methods_and_properties.ipynb](./04_methods_and_properties.ipynb) -- 方法與屬性